**REGRESIÓN LINEAL**

In [25]:
# Importar librerías necesarias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

print("numpy       ", np.__version__)
print("pandas      ", pd.__version__)
print("matplotlib  ", __import__("matplotlib").__version__)
print("seaborn     ", sns.__version__)
print("scikit-learn", sklearn.__version__)

numpy        2.1.3
pandas       2.2.3
matplotlib   3.10.0
seaborn      0.13.2
scikit-learn 1.6.1


In [26]:
# crear dataframe
df = pd.read_csv("https://raw.githubusercontent.com/husseinlopez/icd2026/main/datos/winequality-red.csv")
df.columns = df.columns.str.replace(" ", "_")

print(f"{df.shape[0]} vinos × {df.shape[1]} columnas")
df.head()

1599 vinos × 12 columnas


,fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide,total_sulfur_dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


**REVISIÓN GENERAL**

In [27]:
n_filas, n_columnas = df.shape
n_numericas = df.select_dtypes(include="number").shape[1]
n_no_numericas = df.select_dtypes(exclude="number").shape[1]

print(f"Registros:               {n_filas:>5}")
print(f"Atributos:               {n_columnas:>5}")
print(f"  numéricos:             {n_numericas:>5}")
print(f"  no numéricos:          {n_no_numericas:>5}")
print(f"Duplicados exactos:      {df.duplicated().sum():>5}")
print(f"Registros con faltantes: {df.isna().any(axis=1).sum():>5}")

Registros:                1599
Atributos:                  12
  numéricos:                12
  no numéricos:              0
Duplicados exactos:        240
Registros con faltantes:     0


Duplicados:

Se eliminan los duplicados exactos para que un mismo vino no quede a la vez en entrenamiento y en prueba. Se trabaja sobre una copia para conservar el original.

In [28]:
df_limpio = df.copy()
df_limpio = df_limpio.drop_duplicates().reset_index(drop=True)

print(f"Antes:   {len(df)} vinos")
print(f"Después: {len(df_limpio)} vinos")

Antes:   1599 vinos
Después: 1359 vinos


Antes de partir los datos, se revisa cómo se reparten las calificaciones de quality.

In [29]:
df_limpio["quality"].value_counts().sort_index()

,count
quality,
3,10
4,53
5,577
6,535
7,167
8,17


**REGRESIÓN**



La partición es 80/20 y está estratificada por quality, para que todas las calificaciones queden representadas en prueba en la misma proporción. Esta misma partición se reutilizará en la clasificación.

In [30]:
variable_objetivo = "quality"

In [31]:
from sklearn.model_selection import train_test_split

X = df_limpio.drop(columns=variable_objetivo)
y = df_limpio[variable_objetivo]

X_entrena, X_prueba, y_entrena, y_prueba = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

print(f"Entrenamiento: {len(X_entrena)} vinos")
print(f"Prueba:        {len(X_prueba)} vinos")

Entrenamiento: 1087 vinos
Prueba:        272 vinos


In [32]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

modelo = LinearRegression().fit(X_entrena, y_entrena)

pred_modelo = modelo.predict(X_prueba)
pred_promedio = np.full(len(y_prueba), y_entrena.mean())

pd.DataFrame({
    "RMSE": [np.sqrt(mean_squared_error(y_prueba, pred_promedio)),
             np.sqrt(mean_squared_error(y_prueba, pred_modelo))],
    "R²": [r2_score(y_prueba, pred_promedio),
           r2_score(y_prueba, pred_modelo)],
}, index=["promedio (línea base)", "regresión lineal"]).round(3)

,RMSE,R²
promedio (línea base),0.819,-0.000
regresión lineal,0.652,0.366


Regresión sin estratificación, solo para comparar resultados.

In [33]:
X_entrena_SE, X_prueba_SE, y_entrena_SE, y_prueba_SE = train_test_split(
    X, y, test_size=0.2, random_state=42)

print(f"Entrenamiento: {len(X_entrena_SE)} vinos")
print(f"Prueba:        {len(X_prueba_SE)} vinos")

Entrenamiento: 1087 vinos
Prueba:        272 vinos


In [34]:
pd.DataFrame({
    "entrena (estratificado)": y_entrena.value_counts(normalize=True),
    "prueba (estratificado)": y_prueba.value_counts(normalize=True),
    "entrena (sin estratificar)": y_entrena_SE.value_counts(normalize=True),
    "prueba (sin estratificar)": y_prueba_SE.value_counts(normalize=True),
}).sort_index().mul(100).round(1)

,entrena (estratificado),prueba (estratificado),entrena (sin estratificar),prueba (sin estratificar)
quality,,,,
3,0.7,0.7,0.6,1.5
4,3.9,4.0,3.9,4.0
5,42.4,42.6,42.0,44.1
6,39.4,39.3,39.7,37.9
7,12.3,12.1,12.5,11.4
8,1.3,1.1,1.3,1.1


Con esta semilla, partir sin estratificar cambia poco la distribución. Las diferencias más grandes están en la calificación 3 y en la 5. Con estratificación, ninguna calificación difiere más de 0.2 puntos.

Regresión con conjunto sin estratificar:

In [37]:
modelo_SE = LinearRegression().fit(X_entrena_SE, y_entrena_SE)

pred_modelo_SE = modelo_SE.predict(X_prueba_SE)
pred_promedio_SE = np.full(len(y_prueba_SE), y_entrena_SE.mean())
print('SIN ESTRATIFICAR')
pd.DataFrame({
    "RMSE": [np.sqrt(mean_squared_error(y_prueba_SE, pred_promedio_SE)),
             np.sqrt(mean_squared_error(y_prueba_SE, pred_modelo_SE))],
    "R²": [r2_score(y_prueba_SE, pred_promedio_SE),
           r2_score(y_prueba_SE, pred_modelo_SE)],
}, index=["promedio (línea base)", "regresión lineal"]).round(3)

SIN ESTRATIFICAR


,RMSE,R²
promedio (línea base),0.844,-0.006
regresión lineal,0.657,0.392


Sin estratificar, el modelo falla casi igual (RMSE de 0.657 contra 0.652). R² sube de 0.366 a 0.392. No es que prediga mejor, sino que en esa prueba adivinar con el promedio falla más (0.844 contra 0.819), y el R² mide cuánto mejora el modelo frente a esa referencia.

**CLASIFICACIÓN**

Se define bueno = 1 si quality ≥ 7 y 0 en otro caso. La columna se obtiene de los datos estratificados.

In [38]:
bueno_entrena = (y_entrena >= 7).astype(int)
bueno_prueba = (y_prueba >= 7).astype(int)

print(f"Buenos en entrenamiento: {bueno_entrena.sum()} de {len(bueno_entrena)} ({bueno_entrena.mean():.1%})")
print(f"Buenos en prueba:        {bueno_prueba.sum()} de {len(bueno_prueba)} ({bueno_prueba.mean():.1%})")

Buenos en entrenamiento: 148 de 1087 (13.6%)
Buenos en prueba:        36 de 272 (13.2%)


La recta predice números, no clases: si ŷ ≥ 0.5 el vino se clasifica como bueno. Se compara con la línea base de clasificación, que es predecir siempre «no bueno», y se cuentan las ŷ que quedan fuera de [0, 1].

In [39]:
from sklearn.metrics import accuracy_score

recta = LinearRegression().fit(X_entrena, bueno_entrena)

y_gorro = recta.predict(X_prueba)              # números, no clases
clase = (y_gorro >= 0.5).astype(int)           # umbral: ŷ ≥ 0.5 → bueno
siempre_no = np.zeros(len(bueno_prueba), dtype=int)
fuera = ((y_gorro < 0) | (y_gorro > 1)).sum()

print(f"Aciertos de la recta:              {accuracy_score(bueno_prueba, clase):.1%}")
print(f"Aciertos diciendo siempre «no»:    {accuracy_score(bueno_prueba, siempre_no):.1%}")
print(f"ŷ fuera de [0, 1]:                 {fuera} de {len(y_gorro)}")

Aciertos de la recta:              87.1%
Aciertos diciendo siempre «no»:    86.8%
ŷ fuera de [0, 1]:                 66 de 272


La recta acierta en el 87.1 % de los vinos de prueba, apenas por encima del 86.8 % de decir siempre «no bueno». Además, 66 de sus 272 predicciones quedan fuera de [0, 1], así que no pueden leerse como probabilidad de que un vino sea bueno.

**CONCLUSIÓN**

En la regresión, quality se trató como número y la recta sí aportó. Su error bajó de 0.82 a 0.65 puntos frente a predecir el promedio, con un R² de 0.366. En la clasificación cambian la variable objetivo y la forma de evaluar, ahora hay dos clases muy desbalanceadas. Con la misma recta, el porcentaje de aciertos supera a decir siempre «no bueno» por un solo vino. A la recta le falla que no está hecha para clasificar, minimiza la distancia a los ceros y unos en lugar de modelar la probabilidad de cada clase, por eso 66 de sus predicciones salen de [0, 1]. Además, con una clase minoritaria, el porcentaje de aciertos puede verse alto sin que el modelo detecte casi ningún vino bueno.

**Declaración de uso de IA**

Este notebook fue elaborado con apoyo de Claude (Anthropic) para resolver dudas sobre Python y entrenamiento del modelo. La interpretación de los resultados y la revisión final son responsabilidad del autor.